HARRY GRAY 25% DATA HANDLING:
Sections:
1. Check File intergrity, using threading to reduce processing time. This checks for empty files or corrupt file.(A much easier option would be comparing md5 hashes)
2. JSON to CSV, Conversion
3. Prasing Outliers
4. Final CSV, Ready for Next Assesment

In [ ]:
from concurrent.futures import ThreadPoolExecutor # threading
from tqdm import tqdm ## progress bar
import os # file handling

## Change this to the correct directory this is mine however:
DATA = r"C:\Users\Haza\Desktop\AMT6004MX\acousticbrainz-mediaeval-train"

## gets all the files within the directory and sub directories
files = [os.path.join(root, f)
             for root, _, files in os.walk(DATA)
             for f in files]

valid_files = []
invalid_files = []
THREADS = 16 # number of threads to be used or "workers"

In [ ]:
## Try and except is to catch any errors, the code opens the files in binary mode 'rb' and reads the first 1024 bytes -
## if the file is readable this is a good indication of integrity. Returning the file path if the file checks out,
## otherwise it retunrs false indicating a potenital issue with the file.
## its important that only 1024 bytes are read as the database is large.
def check_file(file_path):
    try:
        with open(file_path, 'rb') as f:
            f.read(1024)
        return True, file_path
    except:
        return False, file_path

## This creates a pool of "workers", THREADS is the number of "workers" (threads) to use, the greater the value the faster it can complete
## but this does consume more resources. "As executor" is used so it can be assigned to a vairable.
## "for valid, file..." is used to submit files to the "workers" within each thread where the check_file function is called.
## the returned values from check_file true or false. It sorts the files between valid and invalid. tqdm is a progress bar libary. 
with ThreadPoolExecutor(max_workers=THREADS) as executor:
    for valid, file_path in tqdm(executor.map(check_file, files), total=len(files), desc="Progress: "):
        if valid:
            valid_files.append(file_path)
        else:
            invalid_files.append(file_path)

print(f"Valid files: {len(valid_files)}")
print(f"Invalid files: {len(invalid_files)}")

Checking files: 100%|██████████| 182978/182978 [00:07<00:00, 23245.42it/s]


Valid files: 182978
Invalid files: 0


In [ ]:
import csv
import json


## Change this to the correct directory this is mine however:
output_dir = r"JSON_TO_CSV"
os.makedirs(output_dir, exist_ok=True)

for folder_name in sorted(os.listdir(DATA)):
    folder_path = os.path.join(DATA, folder_name)
    if not os.path.isdir(folder_path):
        continue

    csv_file_path = os.path.join(output_dir, f"{folder_name}.csv")

    json_files = [f for f in os.listdir(folder_path) if f.endswith(".json")]

    if not json_files:
        continue

    first_file_path = os.path.join(folder_path, json_files[0])
    try:
        with open(first_file_path, "r", encoding="utf-8") as f:
            first_data = json.load(f)

            if isinstance(first_data, list):
                first_data = first_data[0]
            headers = list(first_data.keys())
    except Exception as e:
        print(f"Error {first_file_path}: {e}")
        continue

    with open(csv_file_path, "w", newline="", encoding="utf-8") as csv_f:
        writer = csv.DictWriter(csv_f, fieldnames=headers)
        writer.writeheader()

        for json_file in tqdm(json_files, desc=f"Processing {folder_name}"):
            json_path = os.path.join(folder_path, json_file)
            try:
                with open(json_path, "r", encoding="utf-8") as f:
                    data = json.load(f)
                    if isinstance(data, list):
                        for item in data:
                            writer.writerow({k: item.get(k, None) for k in headers})
                    else:
                        writer.writerow({k: data.get(k, None) for k in headers})
            except Exception as e:
                print(f"Error reading {json_file}: {e}")

    print(f"Created CSV for folder {folder_name}: {csv_file_path}")


Processing folder 40: 100%|██████████| 5726/5726 [00:17<00:00, 333.50it/s]


Created CSV for folder 40: JSON_TO_CSV\40.csv


Processing folder 41: 100%|██████████| 5748/5748 [00:55<00:00, 103.53it/s]


Created CSV for folder 41: JSON_TO_CSV\41.csv


Processing folder 42: 100%|██████████| 5634/5634 [01:08<00:00, 82.16it/s]


...

Created CSV for folder 5d: JSON_TO_CSV\5d.csv


Processing folder 5e: 100%|██████████| 5789/5789 [01:10<00:00, 81.88it/s]


Created CSV for folder 5e: JSON_TO_CSV\5e.csv


Processing folder 5f: 100%|██████████| 5702/5702 [01:09<00:00, 82.10it/s]

Created CSV for folder 5f: JSON_TO_CSV\5f.csv


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import os

plots_dir = "output_plots_combined"
os.makedirs(plots_dir, exist_ok=True)
output_dir = r"JSON_TO_CSV"

for csv_file in os.listdir(output_dir):
    if not csv_file.endswith(".csv"):
        continue

    df = pd.read_csv(os.path.join(output_dir, csv_file))

    def get_danceability(rhythm_str):
        try:
            return ast.literal_eval(rhythm_str).get("danceability")
        except:
            return None

    df["danceability"] = df["rhythm"].apply(get_danceability)
    df = df.dropna(subset=["danceability"])

    fig, axes = plt.subplots(2, 1, figsize=(10, 8))

    sns.lineplot(x=df.index, y=df["danceability"], ax=axes[0])
    axes[0].set_title(f"Danceability - {csv_file}")
    axes[0].set_xlabel("Index")
    axes[0].set_ylabel("Danceability")

    sns.histplot(df["danceability"], kde=True, ax=axes[1])
    axes[1].set_title("Danceability Distribution")


    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, f"{os.path.splitext(csv_file)[0]}_combined.png"))
    plt.close()

print("Complete")


With this graphing plot form matlib, allowing me to identifyied serveral outliers.
![alt text](image.png)

In [ ]:
## Change this to the correct directory this is mine however:
output_filter = r"C:\Users\Haza\Desktop\AMT6004MX\Submission\Code\CSV_Clean"

csv_files = [f for f in os.listdir(output_filter) if f.endswith(".csv")]
if not csv_files:
    print("ERROR: No CSV files")
    exit()

print("CSV files, pick one: ")
for i, f in enumerate(csv_files):
    print(f"{i}: {f}")

file_index = int(input("CSV Number: "))
csv_path = os.path.join(output_filter, csv_files[file_index])
df = pd.read_csv(csv_path)

print("\nColumns in CSV:")
for col in df.columns:
    print(col)

nested_column = input("\nEnter column (eg: rhythm.danceability): ").strip()
parent_col, child_col = nested_column.split(".")

df[parent_col] = df[parent_col].apply(ast.literal_eval)

inputgl = input("Enter condition ('>' or '<'): ").strip()
value = float(input("Enter the value to compare: "))

if inputgl == '>':
    filtered_df = df[df[parent_col].apply(lambda x: x[child_col] > value)]
elif inputgl == '<':
    filtered_df = df[df[parent_col].apply(lambda x: x[child_col] < value)]
else:
    print("Invalid condition. Use '>' or '<'.")
    exit()

filtered_csv_path = os.path.join(output_filter, f"filtered_{csv_files[file_index]}")
filtered_df.to_csv(filtered_csv_path, index=False)

print(f"\nFiltered CSV '{filtered_csv_path}'")
print(f"Rows before: {len(df)}, Rows after: {len(filtered_df)}")


CSV files available:
0: 40.csv
1: 41.csv
2: 42.csv
3: 43.csv
4: 44.csv
5: 45.csv
6: 46.csv
7: 47.csv
8: 48.csv
9: 49.csv
10: 4a.csv
11: 4b.csv
12: 4c.csv
13: 4d.csv
14: 4e.csv
15: 4f.csv
16: 50.csv
17: 51.csv
18: 52.csv
19: 53.csv
20: 54.csv
21: 55.csv
22: 56.csv
23: 57.csv
24: 58.csv
25: 59.csv
26: 5a.csv
27: 5b.csv
28: 5c.csv
29: 5d.csv
30: 5e.csv
31: 5f.csv

Columns in CSV:
tonal
metadata
rhythm
lowlevel

Filtered CSV saved as 'C:\Users\Haza\Desktop\AMT6004MX\Submission\Code\CSV_Clean\filtered_4f.csv'
Rows before: 5668, Rows after: 5667
